<div style="width: 100%; clear: both;">
    <div style="float: left; width: 50%;">
        <img src="http://www.uoc.edu/portal/_resources/common/imatges/marca_UOC/UOC_Masterbrand.jpg" align="left">
    </div>
</div>
<div style="float: right; width: 50%;">
    <p style="margin: 0; padding-top: 22px; text-align:right;">22.521 · Análisis de redes sociales</p>
    <p style="margin: 0; text-align:right;">Grado de Ciencia de Datos Aplicada (<i>Applied Data Science</i>)</p>
    <p style="margin: 0; text-align:right;">Estudios de Informática, Multimedia y Telecomunicación</p>
</div>
<div style="width: 100%; clear: both;"></div>
<div style="width:100%;">&nbsp;</div>

# Práctica: Análisis de la Red de Control del Tráfico Aéreo (FAA)

**22.521 – Análisis de grafos y redes sociales · 2025-2**

---

## Apartado 0 – Justificación de la elección del conjunto de datos

El conjunto de datos elegido es la **red de control del tráfico aéreo de la FAA** (_Federal Aviation Administration_), disponible en KONECT: [http://konect.cc/networks/maayan-faa/](http://konect.cc/networks/maayan-faa/).

Este dataset fue construido a partir de la base de datos de **rutas preferidas** del Centro Nacional de Datos de Vuelo (NFDC) de la FAA de EE.UU. Los nodos representan **aeropuertos y centros de servicio**, y las aristas dirigidas representan **rutas preferidas recomendadas** entre ellos.

**Motivación de la elección:**

- Se trata de una red de **infraestructura real** con alto interés práctico: la estructura de la red de tráfico aéreo determina la resiliencia del sistema ante fallos, la eficiencia de las rutas y la identificación de aeropuertos críticos.
- La red presenta características teóricas muy ricas: es **dirigida** (las rutas tienen sentido), tiene más de **1.000 nodos** (supera el mínimo de 300) y una densidad baja que la convierte en un grafo disperso ideal para el análisis de comunidades y centralidad.
- Conecta directamente con los conceptos de la asignatura: propagación de información (retrasos en cascada), nodos relevantes (hubs de conexión), morfología (¿es small-world?), comunidades (regiones geográficas de tráfico), y detección de nodos críticos (betweenness).
- Es un dataset limpio, bien documentado y de descarga directa desde KONECT.

**Referencia:**
> FAA Air Traffic Control System Command Center. *National Flight Data Center Preferred Routes Database*. KONECT, 2017. http://konect.cc/networks/maayan-faa/

## Apartado 1 – Obtención y limpieza del conjunto de datos

In [ ]:
# Activamos matplotlib para mostrar los plots en el notebook
%matplotlib inline

# Importamos las librerías necesarias
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import community as community_louvain   # python-louvain
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print(f'NetworkX versión: {nx.__version__}')
print(f'Pandas versión:   {pd.__version__}')

In [ ]:
# -----------------------------------------------------------
# Carga del fichero out.maayan-faa
# Formato: dos columnas separadas por espacio (source target)
# La primera línea comienza con '%' → se omite con comment='%'
# -----------------------------------------------------------
df_raw = pd.read_csv(
    'out.maayan-faa',
    sep=' ',
    comment='%',
    header=None,
    names=['source', 'target'],
    usecols=[0, 1]
)

print('=== DATOS BRUTOS ===')
print(f'Filas (aristas): {len(df_raw)}')
print(f'Nodos únicos source: {df_raw["source"].nunique()}')
print(f'Nodos únicos target: {df_raw["target"].nunique()}')
print(f'Total nodos: {len(set(df_raw["source"]) | set(df_raw["target"]))}')
df_raw.head(10)

In [ ]:
# -----------------------------------------------------------
# Limpieza del dataset
#
# Decisiones tomadas:
# 1. Eliminar filas con valores nulos (no hay ninguna, pero
#    se verifica por robustez).
# 2. Convertir los identificadores a entero.
# 3. Eliminar autoenlaces (source == target): el dataset
#    contiene 2 autoenlaces (etiquetados '#loop' en el meta).
#    Un aeropuerto con ruta hacia sí mismo no aporta información
#    útil al análisis de la red y distorsionaría métricas de
#    clustering y centralidad.
# -----------------------------------------------------------

# Paso 1: eliminar nulos
print(f'Filas antes de dropna: {len(df_raw)}')
df_raw = df_raw.dropna()
print(f'Filas tras dropna:     {len(df_raw)}')

# Paso 2: convertir a entero
df_raw['source'] = df_raw['source'].astype(int)
df_raw['target'] = df_raw['target'].astype(int)

# Paso 3: detectar y eliminar autoenlaces
selfloops = df_raw[df_raw['source'] == df_raw['target']]
print(f'\nAutoenlaces detectados: {len(selfloops)}')
print(selfloops)

df = df_raw[df_raw['source'] != df_raw['target']].copy()
print(f'\nAristas tras limpieza: {len(df)}')

In [ ]:
# Estadísticas básicas del dataset limpio
print('=== DATASET LIMPIO ===')
print(f'Aristas:      {len(df)}')
print(f'Nodos únicos: {len(set(df["source"]) | set(df["target"]))}')
print()
print('Distribución de grado saliente (out-degree bruto):')
print(df.groupby('source')['target'].count().describe())

## Apartado 2 – Construcción y visualización del grafo

### Justificación de la elección de herramientas: NetworkX + Gephi

Se utiliza **NetworkX** para toda la parte de carga, limpieza, construcción y cálculo de métricas. NetworkX es la librería estándar de Python para el análisis de redes complejas y permite un control total sobre el proceso. Sin embargo, como indican los propios materiales del curso (*NetworkX Draw IO*), su módulo de visualización no es su objetivo principal y puede quedar eliminado en futuras versiones.

Por ello, se combina con **Gephi**, que es la herramienta dedicada a la visualización de grafos. Gephi permite aplicar layouts avanzados como **ForceAtlas2**, colorear nodos por atributo de comunidad, ajustar el tamaño de los nodos proporcionalmente a métricas de centralidad y exportar imágenes de alta calidad vía la pestaña de previsualización. Esta combinación permite cubrir tanto el análisis riguroso (NetworkX) como la visualización profesional (Gephi).

In [ ]:
# -----------------------------------------------------------
# Construcción del grafo dirigido (DiGraph)
# Justificación: el dataset es asimétrico (asym unweighted
# según el fichero out), lo que indica que las rutas tienen
# dirección. El campo meta confirma 'category: Infrastructure'
# y 'relationship-names: preferred route', por lo que un
# DiGraph es la representación natural.
# -----------------------------------------------------------
G = nx.from_pandas_edgelist(
    df, 'source', 'target',
    create_using=nx.DiGraph()
)

print('=== PROPIEDADES BÁSICAS DEL GRAFO ===')
print(f'Nodos:               {G.number_of_nodes()}')
print(f'Aristas:             {G.number_of_edges()}')
print(f'Densidad:            {nx.density(G):.6f}')
print(f'¿Es conexo (débil)?: {nx.is_weakly_connected(G)}')
print(f'Comp. débiles:       {nx.number_weakly_connected_components(G)}')
print(f'Comp. fuertes:       {nx.number_strongly_connected_components(G)}')

In [ ]:
# -----------------------------------------------------------
# Histogramas de grado
# -----------------------------------------------------------
in_deg  = dict(G.in_degree())
out_deg = dict(G.out_degree())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

in_vals  = list(in_deg.values())
out_vals = list(out_deg.values())

axes[0].hist(in_vals, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('In-degree'); axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de In-Degree')
axes[0].axvline(np.mean(in_vals), color='red', linestyle='--',
                label=f'Media = {np.mean(in_vals):.2f}')
axes[0].legend()

axes[1].hist(out_vals, bins=30, color='coral', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Out-degree'); axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de Out-Degree')
axes[1].axvline(np.mean(out_vals), color='red', linestyle='--',
                label=f'Media = {np.mean(out_vals):.2f}')
axes[1].legend()

plt.suptitle('Distribución de grados — Red FAA de control del tráfico aéreo',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig1_distribucion_grados.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'In-degree  → min={min(in_vals)}, max={max(in_vals)}, media={np.mean(in_vals):.2f}')
print(f'Out-degree → min={min(out_vals)}, max={max(out_vals)}, media={np.mean(out_vals):.2f}')

In [ ]:
# -----------------------------------------------------------
# Distribución de grados en escala log-log
# -----------------------------------------------------------
grado_all = [v for _, v in G.degree()]
cnt = Counter(grado_all)
x = sorted(cnt.keys()); y = [cnt[k] for k in x]

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(x, y, 'o', color='steelblue', markersize=5, alpha=0.7)
ax.set_xlabel('Grado (escala log)'); ax.set_ylabel('Frecuencia (escala log)')
ax.set_title('Distribución de grados en escala log-log\nAnálisis de ley de potencias',
             fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig5_loglog.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# -----------------------------------------------------------
# Exportar a GEXF y GraphML para Gephi
# -----------------------------------------------------------

# Primero calculamos PageRank y betweenness para incluirlos
# como atributos de nodo visibles en Gephi
bet_cent  = nx.betweenness_centrality(G, normalized=True)
pagerank  = nx.pagerank(G, alpha=0.85)
clos_cent = nx.closeness_centrality(G)

# Detección de comunidades (sobre grafo no dirigido)
G_und = G.to_undirected()
partition = community_louvain.best_partition(G_und, random_state=42)

for n in G.nodes():
    G.nodes[n]['label']       = f'N{n}'
    G.nodes[n]['comunidad']   = int(partition.get(n, -1))
    G.nodes[n]['in_degree']   = int(G.in_degree(n))
    G.nodes[n]['out_degree']  = int(G.out_degree(n))
    G.nodes[n]['betweenness'] = float(bet_cent.get(n, 0))
    G.nodes[n]['pagerank']    = float(pagerank.get(n, 0))
    G.nodes[n]['closeness']   = float(clos_cent.get(n, 0))

nx.write_gexf(G, 'faa_network.gexf')
nx.write_graphml(G, 'faa_network.graphml')

print('✓ faa_network.gexf   → abrir en Gephi')
print('✓ faa_network.graphml → formato estándar')
print()
print('Pasos en Gephi:')
print('  1. File → Open → faa_network.gexf')
print('  2. Layout → ForceAtlas2 (ejecutar ~1 min)')
print('  3. Appearance → Nodes → Color → Partition → comunidad')
print('  4. Appearance → Nodes → Size  → Ranking   → betweenness')
print('  5. Preview → Export PNG')

In [ ]:
# -----------------------------------------------------------
# Visualización con NetworkX (subgrafo top-200 nodos)
# -----------------------------------------------------------
import matplotlib.cm as cm

n_coms   = max(partition.values()) + 1
sizes    = Counter(partition.values())
mod      = community_louvain.modularity(partition, G_und)

deg_total = dict(G.degree())
top200    = sorted(deg_total, key=lambda x: deg_total[x], reverse=True)[:200]
G_sub     = G.subgraph(top200).copy()

pos = nx.spring_layout(G_sub, seed=42, k=0.5)

colors15    = plt.cm.tab20(np.linspace(0, 1, n_coms))
node_colors = [colors15[partition.get(n, 0)] for n in G_sub.nodes()]
node_sizes  = [20 + deg_total.get(n, 1) * 10 for n in G_sub.nodes()]

fig, ax = plt.subplots(figsize=(14, 12))
nx.draw_networkx_edges(G_sub, pos, alpha=0.2, width=0.5,
                       edge_color='gray', arrows=True, arrowsize=5, ax=ax)
nx.draw_networkx_nodes(G_sub, pos, node_color=node_colors,
                       node_size=node_sizes, alpha=0.85, ax=ax)

top10_nodes = sorted(deg_total, key=lambda x: deg_total[x], reverse=True)[:10]
labels      = {n: f'N{n}' for n in top10_nodes if n in G_sub.nodes()}
nx.draw_networkx_labels(G_sub, pos, labels=labels, font_size=7,
                        font_weight='bold', ax=ax)

top5_coms = sorted(sizes.items(), key=lambda x: -x[1])[:5]
handles   = [mpatches.Patch(facecolor=colors15[c],
             label=f'Comunidad {c} ({sz} nodos)') for c, sz in top5_coms]
ax.legend(handles=handles, loc='upper right', fontsize=8)

ax.set_title('Red FAA — Top 200 nodos por grado · Comunidades Louvain', fontsize=12, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('fig3_grafo_comunidades.png', dpi=150, bbox_inches='tight')
plt.show()

## Apartado 3 – Análisis y conclusiones

### 3a. Nodos relevantes

Para la identificación de nodos relevantes se emplean cuatro métricas de centralidad, cuya elección se justifica a continuación:

- **In-degree**: identifica los aeropuertos a los que llegan más rutas preferidas → son los *destinos hub*.
- **Betweenness**: identifica los nodos que actúan de puente entre diferentes partes de la red → aeropuertos cuya eliminación fragmentaría más la conectividad.
- **PageRank**: pondera la importancia de un nodo en función de la importancia de los nodos que le apuntan → similar a la influencia en redes sociales pero aplicado a flujo de tráfico.
- **HITS (Hubs y Authorities)**: HITS distingue entre *hubs* (nodos que señalan a muchos buenos destinos) y *authorities* (nodos muy referenciados). En redes de transporte, los authorities son los grandes aeropuertos centrales y los hubs son los centros de control de rutas.

In [ ]:
# -----------------------------------------------------------
# Métricas de centralidad
# -----------------------------------------------------------
print('Calculando métricas de centralidad...')

# Grado
print(f'\nTop 10 por IN-DEGREE (destinos más solicitados):')
top_in = sorted(in_deg.items(), key=lambda x: x[1], reverse=True)[:10]
for n, v in top_in:
    print(f'  Nodo {n:4d}: in-degree = {v}')

print(f'\nTop 10 por OUT-DEGREE (orígenes más activos):')
top_out = sorted(out_deg.items(), key=lambda x: x[1], reverse=True)[:10]
for n, v in top_out:
    print(f'  Nodo {n:4d}: out-degree = {v}')

In [ ]:
print(f'Top 10 por BETWEENNESS (nodos puente críticos):')
top_bet = sorted(bet_cent.items(), key=lambda x: x[1], reverse=True)[:10]
for n, v in top_bet:
    print(f'  Nodo {n:4d}: betweenness = {v:.4f}')

print(f'\nTop 10 por PAGERANK (influencia en el flujo de tráfico):')
top_pr = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:10]
for n, v in top_pr:
    print(f'  Nodo {n:4d}: pagerank = {v:.4f}')

In [ ]:
# HITS
hits_h, hits_a = nx.hits(G, max_iter=500)

print(f'Top 10 HITS - Hubs (centros que distribuyen rutas):')
top_h = sorted(hits_h.items(), key=lambda x: x[1], reverse=True)[:10]
for n, v in top_h:
    print(f'  Nodo {n:4d}: hub_score = {v:.6f}')

print(f'\nTop 10 HITS - Authorities (destinos más referenciados):')
top_a = sorted(hits_a.items(), key=lambda x: x[1], reverse=True)[:10]
for n, v in top_a:
    print(f'  Nodo {n:4d}: authority_score = {v:.6f}')

In [ ]:
# Visualización comparativa de las métricas
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

top_n = 12

top_in_plot  = sorted(in_deg.items(),   key=lambda x: x[1], reverse=True)[:top_n]
top_bet_plot = sorted(bet_cent.items(), key=lambda x: x[1], reverse=True)[:top_n]
top_pr_plot  = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:top_n]

axes[0].barh([f'N{n}' for n,_ in top_in_plot[::-1]],
             [v for _,v in top_in_plot[::-1]], color='steelblue', alpha=0.85)
axes[0].set_title('Top In-Degree'); axes[0].set_xlabel('In-Degree')

axes[1].barh([f'N{n}' for n,_ in top_bet_plot[::-1]],
             [v for _,v in top_bet_plot[::-1]], color='darkorange', alpha=0.85)
axes[1].set_title('Top Betweenness'); axes[1].set_xlabel('Betweenness')

axes[2].barh([f'N{n}' for n,_ in top_pr_plot[::-1]],
             [v for _,v in top_pr_plot[::-1]], color='seagreen', alpha=0.85)
axes[2].set_title('Top PageRank'); axes[2].set_xlabel('PageRank')

plt.suptitle('Nodos más relevantes según métricas de centralidad — Red FAA',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig2_nodos_relevantes.png', dpi=150, bbox_inches='tight')
plt.show()

### 3b. Morfología de la red
#### i. Nivel de conectividad, polarización y aislamiento

In [ ]:
# -----------------------------------------------------------
# Métricas de red (macro)
# -----------------------------------------------------------

# Radio, diámetro y camino medio sobre la SCC gigante
scc_nodes = max(nx.strongly_connected_components(G), key=len)
G_scc     = G.subgraph(scc_nodes).copy()

print('=== MORFOLOGÍA DE LA RED ===')
print(f'Nodos:                    {G.number_of_nodes()}')
print(f'Aristas:                  {G.number_of_edges()}')
print(f'Densidad:                 {nx.density(G):.6f}')
print(f'Grado medio (in):         {np.mean(list(in_deg.values())):.3f}')
print(f'Grado medio (out):        {np.mean(list(out_deg.values())):.3f}')
print(f'Grado máximo (in):        {max(in_deg.values())}')
print(f'Grado máximo (out):       {max(out_deg.values())}')
print(f'Coef. clustering medio:   {nx.average_clustering(G_und):.4f}')
print(f'Asortatividad:            {nx.degree_assortativity_coefficient(G):.4f}')
print()
print(f'SCC gigante: {G_scc.number_of_nodes()} nodos ({G_scc.number_of_nodes()/G.number_of_nodes()*100:.1f}% del total)')
print(f'Diámetro (SCC):           {nx.diameter(G_scc)}')
print(f'Camino medio (SCC):       {nx.average_shortest_path_length(G_scc):.4f}')
print(f'Comp. débiles:            {nx.number_weakly_connected_components(G)}')
print(f'Comp. fuertes:            {nx.number_strongly_connected_components(G)}')

In [ ]:
# -----------------------------------------------------------
# Comparación con red aleatoria Erdős-Rényi
# para determinar si la red tiene comportamiento small-world
# -----------------------------------------------------------
n = G_und.number_of_nodes()
m = G_und.number_of_edges()
p = 2*m / (n*(n-1))

G_rand = nx.erdos_renyi_graph(n, p, seed=42)
gcc_r  = max(nx.connected_components(G_rand), key=len)
G_rand_gc = G_rand.subgraph(gcc_r).copy()

L_real = nx.average_shortest_path_length(G_und)
C_real = nx.average_clustering(G_und)
L_rand = nx.average_shortest_path_length(G_rand_gc)
C_rand = nx.average_clustering(G_rand_gc)
sigma  = (C_real/C_rand) / (L_real/L_rand)

print('=== COMPARACIÓN CON RED ALEATORIA (Small-World) ===')
print(f'  L_real = {L_real:.4f}   L_rand = {L_rand:.4f}   ratio = {L_real/L_rand:.3f}')
print(f'  C_real = {C_real:.4f}   C_rand = {C_rand:.4f}   ratio = {C_real/C_rand:.1f}x')
print(f'  Coeficiente sigma (small-world) = {sigma:.2f}')
if sigma > 1:
    print('  → La red presenta propiedades de SMALL-WORLD (σ > 1)')
else:
    print('  → La red NO presenta propiedades de small-world')

In [ ]:
# Visualización comparativa small-world
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
cats = ['Red FAA\n(real)', 'Red aleatoria\n(Erdős-Rényi)']

b0 = axes[0].bar(cats, [L_real, L_rand], color=['steelblue','lightgray'], edgecolor='white')
axes[0].set_ylabel('Longitud media caminos'); axes[0].set_title('L (camino medio)')
axes[0].bar_label(b0, fmt='%.3f', padding=2)

b1 = axes[1].bar(cats, [C_real, C_rand], color=['coral','lightgray'], edgecolor='white')
axes[1].set_ylabel('Coef. clustering'); axes[1].set_title('C (clustering)')
axes[1].bar_label(b1, fmt='%.4f', padding=2)

plt.suptitle(f'Comparación Small-World: σ = {sigma:.2f} → comportamiento small-world',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('fig6_smallworld.png', dpi=150, bbox_inches='tight')
plt.show()

#### ii. Comunidades

In [ ]:
# -----------------------------------------------------------
# Detección de comunidades con algoritmo de Louvain
# Justificación: Louvain maximiza la modularidad de manera
# eficiente y es el algoritmo que usa Gephi por defecto.
# Se aplica sobre el grafo no dirigido para garantizar
# simetría en las comunidades.
# -----------------------------------------------------------
print(f'Número de comunidades: {n_coms}')
print(f'Modularidad Q:         {mod:.4f}')
print()
print('Distribución de comunidades:')
for cid, sz in sorted(sizes.items(), key=lambda x: -x[1]):
    bar = '█' * int(sz / 5)
    print(f'  Comunidad {cid:2d}: {sz:3d} nodos ({sz/G_und.number_of_nodes()*100:5.1f}%)  {bar}')

In [ ]:
# Visualización del tamaño de comunidades
fig, ax = plt.subplots(figsize=(10, 5))
colors15 = plt.cm.tab20(np.linspace(0, 1, n_coms))
cids = [f'C{c}' for c, _ in sorted(sizes.items(), key=lambda x: -x[1])]
szs  = [sz     for _, sz in sorted(sizes.items(), key=lambda x: -x[1])]
bars = ax.bar(cids, szs, color=colors15[:len(cids)], edgecolor='white', alpha=0.9)
ax.bar_label(bars, padding=2, fontsize=9)
ax.set_xlabel('Comunidad'); ax.set_ylabel('Número de nodos')
ax.set_title(f'Tamaño de las {n_coms} comunidades (Louvain)  ·  Q = {mod:.4f}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_comunidades.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Descripción de cada comunidad: nodos con mayor betweenness dentro de ella
print('=== DESCRIPCIÓN DE COMUNIDADES ===\n')
for cid in sorted(sizes.keys(), key=lambda x: -sizes[x]):
    nodos_c = [n for n, c in partition.items() if c == cid]
    # Top 3 por betweenness dentro de la comunidad
    top3_bet = sorted(nodos_c, key=lambda n: bet_cent.get(n, 0), reverse=True)[:3]
    top3_ind = sorted(nodos_c, key=lambda n: in_deg.get(n, 0),  reverse=True)[:3]
    print(f'Comunidad {cid:2d} ({sizes[cid]} nodos):')
    print(f'  Top betweenness: {[(n, f"{bet_cent[n]:.4f}") for n in top3_bet]}')
    print(f'  Top in-degree:   {[(n, in_deg[n]) for n in top3_ind]}')
    print()

### 3c. Interacciones interesantes

In [ ]:
# -----------------------------------------------------------
# Nodos con discrepancia betweenness vs in-degree
# Nodos con alto betweenness pero bajo in-degree son 'brokers'
# silenciosos: no son los destinos más populares pero controlan
# el flujo entre regiones de la red.
# -----------------------------------------------------------
import pandas as pd

df_central = pd.DataFrame({
    'in_degree':   pd.Series(in_deg),
    'out_degree':  pd.Series(out_deg),
    'betweenness': pd.Series(bet_cent),
    'pagerank':    pd.Series(pagerank),
    'closeness':   pd.Series(clos_cent),
    'comunidad':   pd.Series(partition)
})

# Normalizar
for col in ['in_degree','betweenness']:
    df_central[col+'_norm'] = (df_central[col] - df_central[col].min()) / \
                              (df_central[col].max() - df_central[col].min())

# Brokers: alto betweenness normalizado, bajo in-degree normalizado
df_central['broker_score'] = df_central['betweenness_norm'] - df_central['in_degree_norm']
brokers = df_central.sort_values('broker_score', ascending=False).head(10)

print('=== TOP 10 BROKERS (alto betweenness, bajo in-degree) ===')
print(brokers[['in_degree','betweenness','pagerank','comunidad','broker_score']].to_string())

In [ ]:
# -----------------------------------------------------------
# Nodos puente entre comunidades
# Un nodo es puente inter-comunitario si tiene vecinos en
# más de una comunidad
# -----------------------------------------------------------
def community_bridge_score(G, partition):
    scores = {}
    for n in G.nodes():
        coms_vecinos = set()
        for nb in list(G.predecessors(n)) + list(G.successors(n)):
            coms_vecinos.add(partition.get(nb, -1))
        scores[n] = len(coms_vecinos)
    return scores

bridge_scores = community_bridge_score(G, partition)
top_bridges   = sorted(bridge_scores.items(), key=lambda x: x[1], reverse=True)[:10]

print('=== TOP 10 NODOS PUENTE INTER-COMUNIDAD ===')
print('(número de comunidades diferentes a las que conectan)')
for n, s in top_bridges:
    print(f'  Nodo {n:4d}: conecta {s} comunidades distintas  '
          f'| betweenness={bet_cent[n]:.4f}  in-deg={in_deg[n]}')

### 3d. Curiosidades

In [ ]:
# -----------------------------------------------------------
# Curiosidad 1: Nodos 'sumidero' y nodos 'fuente'
# En una red de rutas preferidas, los sumideros (out=0)
# son aeropuertos que solo reciben tráfico (destinos terminales)
# y las fuentes (in=0) solo lo emiten.
# -----------------------------------------------------------
fuentes   = [n for n in G.nodes() if in_deg[n] == 0]
sumideros = [n for n in G.nodes() if out_deg[n] == 0]

print(f'Nodos FUENTE (in=0, solo emiten rutas):       {len(fuentes)}')
print(f'Nodos SUMIDERO (out=0, solo reciben rutas):   {len(sumideros)}')
print(f'Nodos AISLADOS (in=0 Y out=0):                '
      f'{len([n for n in G.nodes() if in_deg[n]==0 and out_deg[n]==0])}')

In [ ]:
# -----------------------------------------------------------
# Curiosidad 2: Reciprocidad de las rutas
# ¿Con qué frecuencia una ruta A→B tiene su inversa B→A?
# En el tráfico aéreo real, las rutas de ida no siempre
# coinciden con las de vuelta (efecto jet stream, etc.)
# -----------------------------------------------------------
rec = nx.overall_reciprocity(G)
print(f'Reciprocidad global: {rec:.4f}')
print(f'→ Solo el {rec*100:.1f}% de las rutas tienen su correspondiente ruta de vuelta.')
print(f'→ Esto refleja que las rutas preferidas son ASIMÉTRICAS, probablemente')
print(f'  por la corriente en chorro (jet stream) y la dirección del viento predominante.')

In [ ]:
# -----------------------------------------------------------
# Curiosidad 3: Correlación betweenness vs in-degree
# -----------------------------------------------------------
from scipy import stats

x_vals = [in_deg[n] for n in G.nodes()]
y_vals = [bet_cent[n] for n in G.nodes()]
r, p   = stats.pearsonr(x_vals, y_vals)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_vals, y_vals, alpha=0.3, s=10, color='steelblue')
ax.set_xlabel('In-degree'); ax.set_ylabel('Betweenness')
ax.set_title(f'Correlación In-Degree vs Betweenness\nr = {r:.3f}, p = {p:.2e}',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('fig7_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Correlación de Pearson: r = {r:.3f}')
print('→ In-degree y betweenness están correlacionados pero no idénticamente.')
print('  Esto confirma que hay nodos con alto betweenness y bajo in-degree (brokers).')

In [ ]:
# -----------------------------------------------------------
# Curiosidad 4: k-core del grafo no dirigido
# El k-core es el subgrafo maximal donde todos los nodos
# tienen grado ≥ k. Muestra el 'núcleo duro' de la red.
# -----------------------------------------------------------
for k in [1, 2, 3, 4, 5]:
    core = nx.k_core(G_und, k=k)
    print(f'k={k}-core: {core.number_of_nodes()} nodos, {core.number_of_edges()} aristas')

print()
# El núcleo más profundo contiene los aeropuertos más densamente
# interconectados: el corazón real de la red FAA
core5 = nx.k_core(G_und, k=5)
print(f'Nodos en el 5-core (núcleo duro):')
print(sorted(core5.nodes())[:20], '...')

In [ ]:
# -----------------------------------------------------------
# Resumen final
# -----------------------------------------------------------
print('=' * 55)
print('RESUMEN — RED FAA DE CONTROL DEL TRÁFICO AÉREO (EE.UU.)')
print('=' * 55)
print(f'  Nodos:                  {G.number_of_nodes()}')
print(f'  Aristas:                {G.number_of_edges()}')
print(f'  Densidad:               {nx.density(G):.6f}')
print(f'  Grado medio:            {np.mean(list(in_deg.values())):.2f}')
print(f'  Diámetro (SCC):         {nx.diameter(G_scc)}')
print(f'  Camino medio (SCC):     {nx.average_shortest_path_length(G_scc):.4f}')
print(f'  Clustering medio:       {nx.average_clustering(G_und):.4f}')
print(f'  Small-world sigma:      {sigma:.2f}')
print(f'  Comunidades (Louvain):  {n_coms}  Q={mod:.4f}')
print(f'  Reciprocidad:           {rec:.4f}')
top1_ind = sorted(in_deg.items(),   key=lambda x: x[1], reverse=True)[0]
top1_bet = sorted(bet_cent.items(), key=lambda x: x[1], reverse=True)[0]
print(f'  Hub principal (in-deg): Nodo {top1_ind[0]} (in={top1_ind[1]})')
print(f'  Broker principal (btw): Nodo {top1_bet[0]} (bet={top1_bet[1]:.4f})')
print()
print('Ficheros generados:')
for f in ['faa_network.gexf', 'faa_network.graphml',
          'fig1_distribucion_grados.png', 'fig2_nodos_relevantes.png',
          'fig3_grafo_comunidades.png',   'fig4_comunidades.png',
          'fig5_loglog.png', 'fig6_smallworld.png', 'fig7_correlacion.png']:
    print(f'  {f}')